In [1]:
import json
import time
import torch
import networkx as nx
import numpy as np
import pandas as pd

from difflib import SequenceMatcher
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel

C:\Users\molla\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [5]:
############################################
# CONFIG
############################################

THRESHOLD_MAP = {
    "memory": 0.40,
    "color": 0.30,
    "accessory": 0.40,
    "brand": 0.35,
    "product_product": 0.80
}

DEFAULT_THRESHOLD = 0.40

In [7]:
############################################
# LOAD MODEL
############################################

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

model = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [19]:
############################################
# SAFE COSINE
############################################

def safe_cosine(a, b):
    a = np.array(a)
    b = np.array(b)

    if np.isnan(a).any() or np.isnan(b).any():
        return 0

    return cosine_similarity([a], [b])[0][0]


############################################
# SPAN EMBEDDINGS
############################################

def generate_span_embeddings(text, entities):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    offsets = inputs["offset_mapping"][0]
    inputs.pop("offset_mapping")

    with torch.no_grad():
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state[0]

    for ent in entities:

        token_ids = []

        for i, (start, end) in enumerate(offsets):
            if start >= ent["start"] and end <= ent["end"]:
                token_ids.append(i)

        if not token_ids:
            ent["embedding"] = np.zeros(token_embeddings.shape[1])
        else:
            vec = token_embeddings[token_ids].mean(dim=0)
            ent["embedding"] = vec.numpy()

    return entities


############################################
# DISTANCE
############################################

def span_distance(a, b):
    return max(0, max(a["start"] - b["end"], b["start"] - a["end"]))


def distance_score(a, b):
    d = span_distance(a, b)
    return 1 / (d + 1)


############################################
# BUILD GRAPH
############################################

def build_graph(entities):

    G = nx.DiGraph()

    attributes = []
    products = []

    for e in entities:

        G.add_node(e["text"], label=e["label"])

        if e["label"] == "product_name":
            products.append(e)
        else:
            attributes.append(e)

    # ATTRIBUTE → PRODUCT
    for attr in attributes:
        for prod in products:

            sim = safe_cosine(attr["embedding"], prod["embedding"])
            dist = distance_score(attr, prod)

            G.add_edge(
                attr["text"],
                prod["text"],
                relation="attribute_product",
                weight=sim,
                sim=sim,
                dist_score=dist
            )

    # PRODUCT → PRODUCT
    for i in range(len(products)):
        for j in range(i + 1, len(products)):

            p1 = products[i]
            p2 = products[j]

            sim = safe_cosine(p1["embedding"], p2["embedding"])
            dist = distance_score(p1, p2)

            G.add_edge(p1["text"], p2["text"],
                       relation="product_product",
                       weight=sim, sim=sim, dist_score=dist)

            G.add_edge(p2["text"], p1["text"],
                       relation="product_product",
                       weight=sim, sim=sim, dist_score=dist)

    return G


############################################
# LINK ENTITIES
############################################

def link_entities(text, entities):

    entities = generate_span_embeddings(text, entities)
    G = build_graph(entities)

    results = []
    node_labels = {e["text"]: e["label"] for e in entities}

    for node in G.nodes:

        edges = G.out_edges(node, data=True)
        if not edges:
            continue

        best = max(edges, key=lambda x: x[2]["weight"])

        label = node_labels.get(best[0])
        threshold = THRESHOLD_MAP.get(label, DEFAULT_THRESHOLD)

        if best[2]["weight"] >= threshold:

            results.append({
                "attribute": best[0],
                "product": best[1],
                "score": float(round(best[2]["weight"], 3)),
                "sim": float(round(best[2]["sim"], 3)),
                "dist": float(round(best[2]["dist_score"], 3))
            })

    return results


############################################
# FINAL JSON (WITH META)
############################################

def build_final_json(text, entities, linker_results):

    final = {"intent": "product_search", "entities": []}

    product_map = {}
    id_counter = 1

    for ent in entities:
        if ent["label"] == "product_name":

            obj = {
                "id": id_counter,
                "product_name": ent["text"],
                "is_main_product": ent.get("is_main_product", False),
                "meta": {}
            }

            product_map[ent["text"]] = obj
            final["entities"].append(obj)
            id_counter += 1

    for link in linker_results:

        attr = link["attribute"]
        prod = link["product"]

        attr_label = None
        for e in entities:
            if e["text"] == attr:
                attr_label = e["label"]

        if attr_label == "accessory":

            parent_id = product_map[prod]["id"]

            final["entities"].append({
                "id": id_counter,
                "product_name": attr,
                "is_main_product": False,
                "is_related_to": parent_id
            })

            id_counter += 1

        elif attr_label not in ["product_name"]:

            product_map[prod][attr_label] = attr

            product_map[prod]["meta"][attr_label] = {
                "score": link["score"],
                "sim": link["sim"],
                "dist": link["dist"]
            }

    return json.dumps(final)


############################################
# PIPELINE
############################################

def run_pipeline(text, entities):

    product_like = [e for e in entities if e["label"] in ["product_name", "accessory"]]

    if len(product_like) == 1:
        return build_final_json(text, entities, [])

    linker_results = link_entities(text, entities)

    return build_final_json(text, entities, linker_results)

In [21]:
############################################
# FUZZY MATCH
############################################

def fuzzy_match(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


############################################
# EXTRACT MAP
############################################

def extract_product_map(data):

    product_map = {}

    for ent in data["entities"]:

        pname = ent["product_name"].lower()

        attrs = {}
        meta = ent.get("meta", {})

        for k, v in ent.items():
            if k not in ["id", "product_name", "is_main_product", "is_related_to", "meta"]:
                attrs[k] = v.lower()

        product_map[pname] = {
            "attributes": attrs,
            "meta": meta
        }

    return product_map


############################################
# EVALUATION
############################################

def evaluate(csv_path, attribute_list):

    df = pd.read_csv(csv_path)
    outputs = {attr: [] for attr in attribute_list}

    for _, row in df.iterrows():

        query = row["query"]
        ner = json.loads(row["ner_entities"])
        gt = json.loads(row["gt"])

        pred = json.loads(run_pipeline(query, ner))

        gt_map = extract_product_map(gt)
        pred_map = extract_product_map(pred)

        for attr in attribute_list:

            for gt_prod, gt_data in gt_map.items():

                gt_val = gt_data["attributes"].get(attr)

                matched_pred = None

                for pred_prod in pred_map:
                    if fuzzy_match(pred_prod, gt_prod) >= 0.75:
                        matched_pred = pred_prod
                        break

                if not matched_pred:
                    status = "FN_product_name"
                    pred_val = None
                    score = sim = dist = 0

                else:
                    pred_attrs = pred_map[matched_pred]["attributes"]
                    meta = pred_map[matched_pred]["meta"]

                    pred_val = pred_attrs.get(attr)
                    meta_info = meta.get(attr, {})

                    score = meta_info.get("score", 0)
                    sim = meta_info.get("sim", 0)
                    dist = meta_info.get("dist", 0)

                    if gt_val is None and pred_val is None:
                        continue
                    elif gt_val is not None and pred_val is None:
                        status = "FN"
                    elif gt_val is None and pred_val is not None:
                        status = "FP"
                    elif gt_val == pred_val:
                        status = "TP"
                    else:
                        status = "FP"

                outputs[attr].append({
                    "query": query,
                    "product_name": gt_prod,
                    "gt_value": gt_val,
                    "pred_value": pred_val,
                    "score": score,
                    "sim": sim,
                    "dist": dist,
                    "status": status
                })

    for attr, rows in outputs.items():
        pd.DataFrame(rows).to_csv(f"{attr}.csv", index=False)
        print(f"{attr}.csv saved")

In [25]:
############################################
# MAIN
############################################

def main():
    csv_path = "linker_dataset.csv"
    attribute_list = ["memory","color"]

    print("\nRunning Evaluation Pipeline...\n")

    evaluate(csv_path, attribute_list)

    print("\nDone ✅\n")


############################################
# RUN
############################################

if __name__ == "__main__":
    main()


Running Evaluation Pipeline...

memory.csv saved
color.csv saved

Done ✅

